<a href="https://colab.research.google.com/github/forevertheever/data-engineering-zoomcamp2026/blob/main/Spark_in_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prerequisites
Sign up to https://ngrok.com/ to be able to reach Spark UI

In [1]:
%%capture
!pip install pyspark
!pip install findspark
!pip install pyngrok

In [2]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
        .appName('testColab') \
        .getOrCreate()

# Start a tunnel to access SparkUI

Open a ngrok tunnel to the HTTP server

In [4]:
from pyngrok import ngrok, conf
import getpass

print("Enter your authtoken, which can be copied "
"from https://dashboard.ngrok.com/get-started/your-authtoken")
conf.get_default().auth_token = getpass.getpass()

ui_port = 4040
public_url = ngrok.connect(ui_port).public_url
print(f" * ngrok tunnel \"{public_url}\" -> \"http://127.0.0.1:{ui_port}\"")

Enter your authtoken, which can be copied from https://dashboard.ngrok.com/get-started/your-authtoken
··········


 * ngrok tunnel "https://acesodynous-nonevangelical-shalanda.ngrok-free.dev" -> "http://127.0.0.1:4040"


## Download Yellow Taxi Trip records data, read it in with Spark, and count the number of rows

Data source: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

In [42]:
from pyspark import SparkFiles

file_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet'
spark.sparkContext.addFile(file_url)

df = spark.read.parquet(SparkFiles.get('yellow_tripdata_2025-11.parquet'), header=True)

df.count()
#df.show()
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [43]:
df.repartition(4)

DataFrame[VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double, cbd_congestion_fee: double]

In [44]:
size_bytes = df._jdf.queryExecution().optimizedPlan().stats().sizeInBytes()
size_mb = size_bytes / (1024 * 1024)

print(size_mb)

67.83891201019287


In [45]:
from pyspark.sql import functions as F

trips_nov15 = df.filter(
    (F.col("tpep_pickup_datetime") >= "2025-11-15") &
    (F.col("tpep_pickup_datetime") < "2025-11-16")
)

count = trips_nov15.count()

print("Number of trips on Nov 15:", count)

Number of trips on Nov 15: 162604


In [48]:
longest_trip = df.select(
    (F.timestamp_diff("SECOND",
                      "tpep_pickup_datetime",
                      "tpep_dropoff_datetime") / 3600)
    .alias("trip_hours")
).agg(F.max("trip_hours"))

longest_trip.show()

+-----------------+
|  max(trip_hours)|
+-----------------+
|90.64666666666666|
+-----------------+



In [ ]:
# Optionally put the tunnel down
# ngrok.disconnect(public_url)

In [7]:
zone_url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'
spark.sparkContext.addFile(zone_url)

zones = spark.read.csv(SparkFiles.get('taxi_zone_lookup.csv'), header=True)

zones.count()

265

In [49]:
df_joined = df.join(
    zones,
    df.PULocationID == zones.LocationID,
    "left"
)

In [50]:
least_zone = (
    df_joined
    .groupBy("Zone")
    .count()
    .orderBy("count")
)

least_zone.show(10)

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|Governor's Island...|    1|
|Eltingville/Annad...|    1|
|       Arden Heights|    1|
|       Port Richmond|    3|
|       Rikers Island|    4|
|   Rossville/Woodrow|    4|
|         Great Kills|    4|
| Green-Wood Cemetery|    4|
|         Jamaica Bay|    5|
|         Westerleigh|   12|
+--------------------+-----+
only showing top 10 rows
